# Advanced Activity: Local Models with Ollama

This notebook demonstrates building a RAG application with locally-running open-source models using Ollama.

## Prerequisites

1. Install Ollama: https://ollama.com
2. Pull required models:
   ```bash
   ollama pull mistral  # Fast, 7B parameter model (4.1 GB)
   ollama pull nomic-embed-text  # Fast embedding model (274 MB)
   ```
3. Start Ollama server:
   ```bash
   ollama serve
   ```

## Comparison Dimensions

We'll evaluate:
- **Latency**: Time to first token and total generation time
- **Quality**: Answer accuracy against gold standards
- **Resource Usage**: CPU/Memory requirements
- **Throughput**: Tokens per second
- **Cost**: Infrastructure cost vs. API calls

## Setup

In [ ]:
import os
import getpass
import time
from dotenv import load_dotenv

load_dotenv()

# Check if Ollama is running
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=2)
    if response.status_code == 200:
        print("✓ Ollama server is running")
    else:
        print("⚠️ Ollama server not responding. Run: ollama serve")
except requests.exceptions.ConnectionError:
    print("⚠️ Ollama server not found. Run: ollama serve")
    print("   Then pull models:")
    print("   - ollama pull mistral")
    print("   - ollama pull nomic-embed-text")

## 1. Load & Prepare Documents

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

def _tiktoken_len(text: str) -> int:
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

# Load documents
directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = directory_loader.load()
print(f"Loaded {len(documents)} documents")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=0,
    length_function=_tiktoken_len
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

## 2. Build Local RAG Pipeline with Ollama

In [ ]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Setting up local models...")

# Local embeddings
local_embedding = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)
print("✓ Local embedding model loaded: nomic-embed-text")

# Vector store
local_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=local_embedding,
    location=":memory:",
    collection_name="local_rag",
)
local_retriever = local_vectorstore.as_retriever(search_kwargs={"k": 3})
print("✓ Vector store created")

# Local chat model
local_llm = ChatOllama(
    model="mistral",
    base_url="http://localhost:11434",
    temperature=0,
    num_ctx=4096,  # Context window
    num_predict=512,  # Max output tokens
)
print("✓ Local chat model loaded: mistral")

## 3. Create RAG Chain

In [ ]:
# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use only the provided context to answer questions. If the answer is not in the context, say 'I don't know'."),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])

# Create RAG chain
def create_local_rag(question: str) -> dict:
    """Execute a RAG query with local models and measure latency."""
    start_time = time.time()
    
    # Retrieval
    retrieval_start = time.time()
    docs = local_retriever.invoke(question)
    retrieval_time = time.time() - retrieval_start
    
    context = "\n\n".join([d.page_content for d in docs])
    
    # Generation
    generation_start = time.time()
    chain = rag_prompt | local_llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    generation_time = time.time() - generation_start
    
    total_time = time.time() - start_time
    
    return {
        "question": question,
        "answer": answer,
        "context": docs,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "total_time": total_time,
    }

print("✓ RAG chain created")

## 4. Test with Sample Queries

In [ ]:
test_queries = [
    "What are the recommended vaccinations for kittens?",
    "How often should I feed an adult cat?",
    "What are common signs of parasites in cats?",
]

local_results = []

print("Running local RAG queries...\n")
for i, question in enumerate(test_queries, 1):
    print(f"Query {i}: {question}")
    result = create_local_rag(question)
    local_results.append(result)
    print(f"  Retrieval: {result['retrieval_time']:.2f}s")
    print(f"  Generation: {result['generation_time']:.2f}s")
    print(f"  Total: {result['total_time']:.2f}s")
    print(f"  Answer: {result['answer'][:100]}...\n")

## 5. Latency Comparison: Local vs. Fireworks

In [ ]:
import pandas as pd

# Summarize latency
latency_summary = pd.DataFrame({
    "Metric": ["Retrieval Time", "Generation Time", "Total Time"],
    "Local (Mistral)": [
        f"{pd.Series([r['retrieval_time'] for r in local_results]).mean():.2f}s",
        f"{pd.Series([r['generation_time'] for r in local_results]).mean():.2f}s",
        f"{pd.Series([r['total_time'] for r in local_results]).mean():.2f}s",
    ],
    "Expected (Fireworks)": [
        "0.5-1.0s",  # Typical retrieval
        "1.0-3.0s",  # Generation with network latency
        "1.5-4.0s",  # Total
    ]
})

print("\n⏱️  Latency Comparison")
print(latency_summary.to_string(index=False))

print("\nKey Observations:")
avg_total = pd.Series([r['total_time'] for r in local_results]).mean()
if avg_total < 2:
    print("  ✓ Local models are FASTER (no network latency)")
elif avg_total < 4:
    print("  ✓ Local models are COMPARABLE to Fireworks")
else:
    print("  ⚠️ Local models are SLOWER (check Ollama configuration)")

## 6. Quality Comparison

In [ ]:
print("\n📊 Sample Responses\n")

for i, result in enumerate(local_results, 1):
    print(f"Query {i}: {result['question']}")
    print(f"Local (Mistral):")
    print(f"  {result['answer'][:200]}...")
    print()

print("\n💡 Quality Assessment:")
print("""
Local (Mistral 7B):
  ✓ Decent quality for straightforward questions
  ✓ Good at factual retrieval and summarization
  ✗ May struggle with complex reasoning or nuance
  ✗ Slightly higher error rate on medical/technical content

Recommendation:
  - Use local models for: FAQs, simple queries, retrieval-heavy tasks
  - Use Fireworks/OpenAI for: Complex reasoning, medical accuracy, nuance
""")

## 7. Resource Usage Analysis

In [ ]:
print("""
🖥️  Resource Usage Comparison

LOCAL (Mistral 7B + Nomic Embeddings):
  Model Size:
    - Mistral 7B: ~4.1 GB
    - Nomic Embeddings: ~274 MB
    - Total: ~4.4 GB
  
  Memory Requirements:
    - Loaded models: ~5-6 GB RAM
    - Operating system & other processes: ~2-3 GB
    - Total: ~8 GB RAM minimum (16 GB recommended)
  
  Compute:
    - Runs on CPU (can use GPU if available)
    - Typical latency on Apple Silicon: 1-2s per query
    - Typical latency on CPU: 5-10s per query
  
  Cost:
    - One-time download: Free (open-source)
    - Ongoing cost: Electricity (~5-15W continuous)
    - Per month: ~$1-5 in electricity

FIREWORKS (gpt-oss-20b):
  Model Size:
    - gpt-oss-20b: Hosted by Fireworks
  
  Resource Requirements:
    - Your machine: Minimal (just API calls)
    - Fireworks infrastructure: Managed for you
  
  Compute:
    - Optimized GPU inference (H100, A100)
    - Typical latency: 1-3s per query
  
  Cost:
    - Per token: $0.15/M input + $0.15/M output
    - Example: 200 input + 100 output tokens = $0.000045
    - Per month (1000 queries): ~$15

OPENAI (gpt-4o-mini):
  Resource Requirements:
    - Your machine: Minimal (just API calls)
  
  Compute:
    - Optimized GPU inference
    - Typical latency: 1-2s per query
  
  Cost:
    - Per token: $0.15/M input + $0.60/M output
    - Example: 200 input + 100 output tokens = $0.00009
    - Per month (1000 queries): ~$45
""")

## 8. Trade-offs Analysis

In [ ]:
import pandas as pd

tradeoffs = pd.DataFrame({
    "Factor": [
        "Latency (per query)",
        "Quality",
        "Monthly Cost (1k queries)",
        "Setup Time",
        "Maintenance",
        "Scalability",
        "Privacy",
        "Customization",
    ],
    "Local (Mistral)": [
        "1-10s",
        "Good",
        "$1-5",
        "30 min",
        "High",
        "Hard",
        "Excellent",
        "Excellent",
    ],
    "Fireworks": [
        "1-3s",
        "Good",
        "$15",
        "5 min",
        "Low",
        "Easy",
        "Good",
        "Limited",
    ],
    "OpenAI": [
        "1-2s",
        "Excellent",
        "$45",
        "1 min",
        "None",
        "Very Easy",
        "Fair",
        "None",
    ]
})

print("\n📋 Comprehensive Trade-offs Analysis\n")
print(tradeoffs.to_string(index=False))

## 9. Recommendations by Use Case

In [ ]:
print("""
🎯 Recommendations by Use Case

1️⃣  STARTUPS / MVP PHASE
   → Use Fireworks (gpt-oss-20b)
   Why: Fastest to deploy, reasonable cost, no infrastructure management
   Cost: ~$15-50/month for typical usage

2️⃣  CUSTOMER-FACING APPLICATIONS
   → Use OpenAI (gpt-4o-mini) or Fireworks + better retrieval
   Why: Quality matters more than cost at small scale
   Cost: $30-200/month depending on volume

3️⃣  INTERNAL TOOLS / EXPERIMENTATION
   → Use Local (Mistral) or Fireworks
   Why: Can iterate without API costs
   Cost: $0-30/month

4️⃣  PRIVACY-CRITICAL APPLICATIONS
   → Use Local (Mistral) + Air-gapped infrastructure
   Why: Data never leaves your infrastructure
   Cost: $0/month (just infrastructure)

5️⃣  HIGH-VOLUME PRODUCTION (>10k queries/day)
   → Use Local (Mistral) or dedicated Fireworks endpoint
   Why: Dedicated endpoints have predictable costs and throughput
   Cost: $100-1000/month for dedicated Fireworks

6️⃣  QUALITY-CRITICAL, VARIABLE VOLUME
   → Use semantic routing: Local for simple, OpenAI for complex
   Why: Optimize cost while maintaining quality where needed
   Cost: $20-100/month

HYBRID STRATEGY (Recommended for Production):
  1. Deploy Fireworks (gpt-oss-20b) as default
  2. Monitor retrieval quality and user feedback
  3. Route complex queries to OpenAI
  4. Maintain local models as fallback for infrastructure issues
  5. Re-evaluate quarterly with LangSmith data
""")

## 10. Future Optimizations

In [ ]:
print("""
🚀 Optimization Strategies

TO IMPROVE LOCAL MODEL QUALITY:
  1. Use larger models (13B, 34B variants)
  2. Fine-tune on domain-specific data
  3. Implement retrieval enhancement:
     - Hybrid search (vector + BM25)
     - Query expansion
     - Re-ranking with cross-encoders
  4. Use prompt engineering for better outputs
  5. Implement fallback to better models for uncertain answers

TO IMPROVE LATENCY:
  1. Use GPU acceleration (if available)
  2. Implement response streaming
  3. Cache embeddings for frequent documents
  4. Batch queries for throughput
  5. Use quantized models (4-bit, 8-bit)

TO REDUCE COSTS:
  1. Implement semantic caching (avoid re-processing similar queries)
  2. Use rate limiting and request batching
  3. Monitor token usage with LangSmith
  4. Optimize chunk size and retrieval parameters
  5. Implement local-first RAG with fallback to API

TO IMPROVE RELIABILITY:
  1. Implement circuit breakers for API calls
  2. Use local models as fallback
  3. Monitor latency and error rates
  4. Implement retry logic with exponential backoff
  5. Use multiple providers (multi-cloud strategy)
""")

## Summary

Local models with Ollama offer a compelling alternative to cloud-based LLM APIs:

✅ **Best For:**
- Privacy-critical applications
- Offline/air-gapped deployments
- Cost-sensitive, high-volume use cases
- Experimentation and prototyping
- Custom model fine-tuning

❌ **Not Best For:**
- Complex reasoning tasks
- Medical/legal accuracy requirements
- Rapid deployment/minimal DevOps
- Mobile or serverless deployments

🎓 **Key Insight:**
The future of LLM applications is hybrid: use local models for simple, high-volume tasks and cloud APIs for complex, accuracy-critical queries. Tools like LangSmith enable data-driven decisions about where to route each request.